# Недостающие маршруты из OpenStreetMap

В справочнике организаторов есть геометрия только для маршрутов 1, 5, 7, 11 и 12
(`dataset/README.md:84-86`). Для 17, 25, 26, 28 и 50 остановки и рельсы берём из
OpenStreetMap, а маршруты из справочника оставляем как есть, чтобы опираться
на данные авторов хакатона.

- **Источник:** OpenStreetMap через Overpass API, срез на **25.12.2025**, то есть
  внутри прогнозного периода. Сеть в OSM с тех пор менялась, поэтому берётся
  историческое состояние, а не текущее.
- **Лицензия:** ODbL, на карте нужна подпись «© участники OpenStreetMap».
- **Как пересобрать файл:** `uv run python ml/scripts/fetch_osm_routes.py`.

In [ ]:
import json
from pathlib import Path

import folium
import numpy as np
import pandas as pd

GEOJSON = Path("dataset/external/osm_missing_routes.geojson")
SPRAVOCHNIK = Path("dataset/spravochniki/10_routes.xlsx")
# маршруты из валидаций, dataset/README.md:81
DATA_ROUTES = [1, 5, 7, 11, 12, 17, 25, 26, 28, 50]
PALETTE = [
    "#e6194b",
    "#3cb44b",
    "#4363d8",
    "#f58231",
    "#911eb4",
    "#42d4f4",
    "#f032e6",
    "#9a6324",
    "#469990",
    "#808000",
]
ROUTE_COLORS = dict(zip(DATA_ROUTES, PALETTE, strict=True))
MOSCOW_CENTER = (55.7558, 37.6173)
KM_PER_DEGREE = 111.32

## Что скачал скрипт

In [ ]:
if not GEOJSON.exists():
    raise FileNotFoundError(
        f"Нет {GEOJSON}: запустите uv run python ml/scripts/fetch_osm_routes.py"
    )

collection = json.loads(GEOJSON.read_text(encoding="utf-8"))
collection["metadata"]

In [ ]:
features = pd.DataFrame(
    [{**f["properties"], "geometry": f["geometry"]} for f in collection["features"]]
)
osm_stops = features[features["kind"] == "stop"].assign(
    stop_lon=lambda df: df["geometry"].map(lambda g: g["coordinates"][0]),
    stop_lat=lambda df: df["geometry"].map(lambda g: g["coordinates"][1]),
)
tracks = features[features["kind"] == "track"]

# длина по сумме отрезков рельсов; на широте Москвы градус долготы короче в cos(55.75°)
lon_scale = np.cos(np.radians(MOSCOW_CENTER[0]))
tracks = tracks.assign(
    track_km=tracks["geometry"].map(
        lambda g: sum(
            np.hypot(np.diff(np.array(line)[:, 1]), np.diff(np.array(line)[:, 0]) * lon_scale).sum()
            for line in g["coordinates"]
        )
        * KM_PER_DEGREE
    )
)

summary = osm_stops.groupby(["route", "direction_id"]).agg(
    stops=("stop_sequence", "size"),
    first_stop=("stop_name", "first"),
    last_stop=("stop_name", "last"),
)
summary.join(tracks.set_index(["route", "direction_id"])[["from", "to", "track_km"]]).round(1)

`direction_id` у маршрутов из OSM условный: направления пронумерованы по
возрастанию идентификатора relation в OSM. Куда идёт направление, видно
по `from` и `to`.

## Все 10 маршрутов на одной карте

У маршрутов из справочника линия идёт по прямой между остановками, формы трассы
в справочнике нет. У маршрутов из OSM линия идёт по рельсам. Источник подписан
в легенде.

In [ ]:
official = pd.read_excel(SPRAVOCHNIK, sheet_name="Порядок_с_координатами")
official = official[official["route_short_name"].isin(DATA_ROUTES)].rename(
    columns={"route_short_name": "route"}
)
all_stops = pd.concat(
    [official.assign(source="справочник"), osm_stops.assign(source="OSM")], ignore_index=True
).sort_values(["route", "direction_id", "stop_sequence"])
print("Маршрутов на карте:", sorted(all_stops["route"].unique().tolist()))
print("Без геометрии:", sorted(set(DATA_ROUTES) - set(all_stops["route"])))

In [ ]:
route_map = folium.Map(location=MOSCOW_CENTER, zoom_start=11, tiles=None)
# OSM блокирует тайлы без заголовка Referer, а его нет ни в выводе VS Code,
# ни у html, открытого из файла; CartoDB требует API-ключ. Esri — без обоих
folium.TileLayer("Esri.WorldGrayCanvas", name="Светлая подложка").add_to(route_map)
folium.TileLayer("Esri.WorldStreetMap", name="Улицы с подписями", show=False).add_to(route_map)

for (route, source), route_stops in all_stops.groupby(["route", "source"]):
    color = ROUTE_COLORS[route]
    ends = route_stops[route_stops["direction_id"] == 0]["stop_name"]
    label = (
        f'<span style="color:{color}">●</span> {route}: {ends.iloc[0]} — {ends.iloc[-1]} '
        f"<i>({source})</i>"
    )
    layer = folium.FeatureGroup(name=label).add_to(route_map)

    for direction, trip in route_stops.groupby("direction_id"):
        if source == "OSM":
            track = tracks[(tracks["route"] == route) & (tracks["direction_id"] == direction)]
            lines = [
                [[lat, lon] for lon, lat in line]
                for line in track.iloc[0]["geometry"]["coordinates"]
            ]
        else:
            lines = [trip[["stop_lat", "stop_lon"]].to_numpy().tolist()]
        folium.PolyLine(
            lines,
            color=color,
            weight=4,
            opacity=0.8,
            dash_array=None if direction == 0 else "6 8",
            tooltip=f"Маршрут {route}, направление {direction} ({source})",
        ).add_to(layer)

        terminals = {trip.index[0], trip.index[-1]}
        for stop in trip.itertuples():
            folium.CircleMarker(
                (stop.stop_lat, stop.stop_lon),
                radius=7 if stop.Index in terminals else 4,
                color=color,
                fill=True,
                fill_opacity=0.9,
                tooltip=(
                    f"{stop.stop_name} · маршрут {route}, "
                    f"направление {direction}, остановка №{stop.stop_sequence}"
                ),
            ).add_to(layer)

osm_credit = (
    '<div style="position: fixed; bottom: 24px; left: 8px; z-index: 1000; background: white;'
    ' padding: 2px 6px; font-size: 11px;">'
    "Маршруты 17, 25, 26, 28, 50: © участники OpenStreetMap (ODbL)</div>"
)
route_map.get_root().html.add_child(folium.Element(osm_credit))
folium.LayerControl(collapsed=False).add_to(route_map)
route_map.fit_bounds(all_stops[["stop_lat", "stop_lon"]].agg(["min", "max"]).to_numpy().tolist())
route_map.save("map_all_routes.html")
route_map